# Chapter 6: Building Your First AI Agent

Hands-On: A Multi-Tool Investment Analyst, Adapted from a Real Project

Extracted from: chapter_06_first_agent.md
Source book: Agentic AI: Building AI Agents and Retrieval Systems,
a Masterclass in LLM Agents, RAG, and Production Deployment.

Every block below was verified by direct execution before being
written into the handbook; run this file top to bottom, or copy
out the section you need. Where a step needs an API key
(OPENAI_API_KEY / ANTHROPIC_API_KEY), it is loaded from a local
.env file via python-dotenv, following Chapter 5's own security
discipline, never hardcoded.

## Installation

Run this once per environment before the cells below.

In [ ]:
%pip install -q phidata python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from dotenv import load_dotenv
load_dotenv()  # picks up OPENAI_API_KEY / ANTHROPIC_API_KEY / COHERE_API_KEY from the project's .env


True

**Step 1: Register a tool with more than one named capability.**

In [ ]:
from phi.assistant import Assistant
from phi.llm.openai import OpenAIChat
from phi.tools.yfinance import YFinanceTools

analyst = Assistant(
    llm=OpenAIChat(model="gpt-4o"),
    tools=[YFinanceTools(
        stock_price=True,
        analyst_recommendations=True,
        company_info=True,
        company_news=True,
    )],
    show_tool_calls=True,
)

**Step 2: Give the agent one genuinely multi-step goal, not a single-fact question.**

In [ ]:
query = """
Compare AAPL and MSFT. Cover:
1. Price performance and trend
2. Analyst recommendations
3. Company fundamentals
4. Recent news and developments
Then give a one-paragraph recommendation.
"""

**Step 3: Run the agent and let it decide, on its own, which of the four tool capabilities to call and how many times.**

In [ ]:
# BUG FIX: Assistant.run() streams by default (stream=True), returning a
# generator; `print(response)` on that generator just printed its repr
# (<generator object Assistant._run at ...>) instead of any real content.
# stream=False makes it return the completed string instead.
response = analyst.run(query, stream=False)
print(response)


Running:
 - get_current_stock_price(symbol=AAPL)
 - get_current_stock_price(symbol=MSFT)
 - get_analyst_recommendations(symbol=AAPL)
 - get_analyst_recommendations(symbol=MSFT)
 - get_company_info(symbol=AAPL)
 - get_company_info(symbol=MSFT)
 - get_company_news(symbol=AAPL, num_stories=3)
 - get_company_news(symbol=MSFT, num_stories=3)

### Price Performance and Trend
- **Apple Inc. (AAPL)**: Current stock price is $312.99, with a 52-week range of $225.95 to $344.57. The 50-day average stands at $310.88, while the 200-day average is $281.90, indicating a steady upward trend over the longer term.
- **Microsoft Corp. (MSFT)**: Current stock price is $494.36, with a 52-week range of $349.20 to $553.72. The 50-day average is $423.80, and the 200-day average is $431.10, suggesting a strong recent performance.

### Analyst Recommendations
- **Apple Inc.**: Recent analyst opinions indicate 6 strong buys, 19 buys, 14 holds, 3 sells, and 2 strong sells, maintaining a "buy" recommendation over